In [1]:
import pandas as pd

# 1. Load the raw dataset
df = pd.read_csv('../data/superstore_raw.csv', encoding='latin1')

# 2. Display metadata
print("=== DATASET OVERVIEW ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

# 3. Clean column names
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_').str.replace('-', '_')

# 4. Standardize date columns
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# 5. Check for duplicates and missing values
print("\nDuplicates Count:", df.duplicated().sum())
print("\nMissing Values Count:\n", df.isnull().sum())

# 6. Create required calculated fields
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['profit_margin'] = (df['profit'] / df['sales']) * 100

# 7. Export cleaned version
df.to_csv('../data/superstore_cleaned.csv', index=False)
print("\nCleaned file successfully exported to 'data/superstore_cleaned.csv'!")

=== DATASET OVERVIEW ===
Total Rows: 9994
Total Columns: 21

Duplicates Count: 0

Missing Values Count:
 row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

Cleaned file successfully exported to 'data/superstore_cleaned.csv'!


In [2]:
# 6. Check data types and summary statistics
print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== DATE RANGE ===")
print(f"Order Date: {df['order_date'].min()} to {df['order_date'].max()}")
print(f"Ship Date:  {df['ship_date'].min()} to {df['ship_date'].max()}")

print("\n=== NUMERIC SUMMARY ===")
print(df[['sales', 'quantity', 'discount', 'profit']].describe().T[['min', 'mean', 'max']])

=== DATA TYPES ===
row_id                    int64
order_id                    str
order_date       datetime64[us]
ship_date        datetime64[us]
ship_mode                   str
customer_id                 str
customer_name               str
segment                     str
country                     str
city                        str
state                       str
postal_code               int64
region                      str
product_id                  str
category                    str
sub_category                str
product_name                str
sales                   float64
quantity                  int64
discount                float64
profit                  float64
year                      int32
month                     int32
year_month                  str
profit_margin           float64
dtype: object

=== DATE RANGE ===
Order Date: 2014-01-03 00:00:00 to 2017-12-30 00:00:00
Ship Date:  2014-01-07 00:00:00 to 2018-01-05 00:00:00

=== NUMERIC SUMMARY ===
            

In [3]:
# 7. Convert postal_code to string/object
df['postal_code'] = df['postal_code'].astype(str)

# 8. Save cleaned dataset to the data directory
df.to_csv('../data/superstore_cleaned.csv', index=False)

print("Cleaned dataset successfully saved to '../data/superstore_cleaned.csv'!")

Cleaned dataset successfully saved to '../data/superstore_cleaned.csv'!


In [4]:
# === 1. KEY PERFORMANCE INDICATORS ===
total_sales = df['sales'].sum()
total_profit = df['profit'].sum()
total_orders = df['order_id'].nunique()
overall_margin = (total_profit / total_sales) * 100

print("=== 1. KEY PERFORMANCE INDICATORS ===")
print(f"Total Sales:           ${total_sales:,.2f}")
print(f"Total Profit:          ${total_profit:,.2f}")
print(f"Total Unique Orders:   {total_orders:,}")
print(f"Overall Profit Margin: {overall_margin:.2f}%\n")

# === 2. PERFORMANCE BY CATEGORY ===
category_perf = df.groupby('category').agg(
    Total_Sales=('sales', 'sum'),
    Total_Profit=('profit', 'sum'),
    Order_Count=('order_id', 'nunique')
).reset_index()

category_perf['Profit_Margin_%'] = (category_perf['Total_Profit'] / category_perf['Total_Sales']) * 100

print("=== 2. PERFORMANCE BY CATEGORY ===")
print(category_perf.round(2).to_string(index=False))
print("\n")

# === 3. PERFORMANCE BY REGION ===
region_perf = df.groupby('region').agg(
    Total_Sales=('sales', 'sum'),
    Total_Profit=('profit', 'sum'),
    Order_Count=('order_id', 'nunique')
).reset_index()

region_perf['Profit_Margin_%'] = (region_perf['Total_Profit'] / region_perf['Total_Sales']) * 100

print("=== 3. PERFORMANCE BY REGION ===")
print(region_perf.round(2).to_string(index=False))

=== 1. KEY PERFORMANCE INDICATORS ===
Total Sales:           $2,297,200.86
Total Profit:          $286,397.02
Total Unique Orders:   5,009
Overall Profit Margin: 12.47%

=== 2. PERFORMANCE BY CATEGORY ===
       category  Total_Sales  Total_Profit  Order_Count  Profit_Margin_%
      Furniture    741999.80      18451.27         1764             2.49
Office Supplies    719047.03     122490.80         3742            17.04
     Technology    836154.03     145454.95         1544            17.40


=== 3. PERFORMANCE BY REGION ===
 region  Total_Sales  Total_Profit  Order_Count  Profit_Margin_%
Central    501239.89      39706.36         1175             7.92
   East    678781.24      91522.78         1401            13.48
  South    391721.90      46749.43          822            11.93
   West    725457.82     108418.45         1611            14.94


# Week 1 EDA Summary: Sample - Superstore

## Data Quality & Baseline Metrics
- **Dataset Volume**: 9,994 transaction records cleaned with 0 duplicates and 0 missing values.
- **Total Revenue**: $2,297,200.86
- **Total Profit**: $286,397.02 (12.47% overall profit margin across 5,009 unique orders).

## Strategic Findings
1. **Category Performance**: Technology leads profitability (17.40% margin), whereas Furniture suffers from compressed margins (2.49%).
2. **Geographic Distribution**: The West region generates the highest net profit ($108.4k), while the Central region produces the lowest margin (7.92%).